# Lab 4: AI Agent Image Classifier

## Building an AI Agent that Classifies Images

In this lab we wrap a trained CNN inside an **AI agent** -- a system that can perceive inputs, reason about them, and take actions. The agent uses a CIFAR-10 classifier as its core "tool" and provides natural language responses about what it sees.

**Backend:** Keras 3 with PyTorch  
**Dataset:** CIFAR-10

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
# Backend setup -- must be set before importing Keras
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import json
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")

## Step 1: What is an AI Agent?

An **AI agent** is a system that operates through a continuous loop of:

1. **Perception** -- The agent receives input from its environment (e.g., an image).
2. **Reasoning** -- The agent processes the input using its internal model and interprets the results.
3. **Action** -- The agent produces an output (e.g., a classification, a natural language explanation, or a decision).

```
           +-------------+
           |  Environment |
           +------+------+
                  |
           Perception (image input)
                  |
           +------v------+
           |    Agent     |
           |  - Model     |
           |  - Tools     |
           |  - Logic     |
           +------+------+
                  |
           Reasoning (inference + interpretation)
                  |
           Action (classification response)
                  |
           +------v------+
           |   Output     |
           +-------------+
```

In this lab, our agent has a single tool -- a CNN image classifier -- and it uses that tool to answer questions about images.

## Step 2: Load the Trained Model

We will train a quick CIFAR-10 CNN here so this notebook is self-contained. If you have the model from Lab 3, you can load it instead by uncommenting the alternative cell below.

In [ ]:
# CIFAR-10 class names
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Train a quick CNN (or load from Lab 3)
# To load from Lab 3 instead, uncomment the next two lines and comment out the training block:
# model = keras.saving.load_model("../lab-3-vibe-coding-cnn/best_cifar10_model.keras")
# print("Loaded model from Lab 3.")

# Quick CNN for this lab
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    keras.layers.Conv2D(32, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(128, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="agent_cnn")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("Training a quick CNN for the agent...")
model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test),
    verbose=1,
)

_, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nModel test accuracy: {test_acc:.4f}")

## Step 3: Build the classify_image Tool

A **tool** is a function that the agent can call. Our first tool takes an image, runs it through the CNN, and returns the top-3 predictions with confidence scores.

In [ ]:
def classify_image(image, model=model, class_names=CLASS_NAMES, top_k=3):
    """
    Classify a single image using the CNN model.

    Args:
        image: numpy array of shape (32, 32, 3) with values in [0, 1].
        model: the trained Keras model.
        class_names: list of class name strings.
        top_k: number of top predictions to return.

    Returns:
        dict with keys:
            - "predictions": list of dicts with "class" and "confidence"
            - "top_class": the highest-confidence class name
            - "top_confidence": the highest confidence score
    """
    # Ensure correct shape: (1, 32, 32, 3)
    if image.ndim == 3:
        image = np.expand_dims(image, axis=0)

    # Run inference
    predictions = model.predict(image, verbose=0)[0]

    # Get top-k indices sorted by confidence (descending)
    top_indices = np.argsort(predictions)[::-1][:top_k]

    results = {
        "predictions": [
            {
                "class": class_names[idx],
                "confidence": float(predictions[idx]),
            }
            for idx in top_indices
        ],
        "top_class": class_names[top_indices[0]],
        "top_confidence": float(predictions[top_indices[0]]),
    }

    return results


# Quick test
sample_result = classify_image(x_test[0])
print("Sample classification result:")
print(json.dumps(sample_result, indent=2))

## Step 4: Build the Agent Class

The `ImageClassifierAgent` class ties everything together:
- It loads the model and registers available tools.
- It can process requests by selecting the appropriate tool.
- It formats results into natural language responses.

In [ ]:
class ImageClassifierAgent:
    """An AI agent that classifies images using a CNN model."""

    def __init__(self, model, class_names):
        """
        Initialise the agent.

        Args:
            model: a trained Keras model.
            class_names: list of class name strings.
        """
        self.model = model
        self.class_names = class_names

        # Register tools -- mapping from tool name to callable
        self.tools = {
            "classify_image": self._classify_image_tool,
        }

    def _classify_image_tool(self, image):
        """Internal wrapper around the classify_image function."""
        return classify_image(image, model=self.model, class_names=self.class_names)

    def process_request(self, request, image=None):
        """
        Parse a request and dispatch to the appropriate tool.

        Args:
            request: a string describing what the user wants (e.g., "classify this image").
            image: optional numpy array of the image to process.

        Returns:
            A string response from the agent.
        """
        request_lower = request.lower()

        # Simple keyword-based routing
        if any(keyword in request_lower for keyword in ["classify", "identify", "what is", "recognise", "recognize"]):
            if image is None:
                return "I need an image to classify. Please provide one."
            return self.classify(image)

        # List available tools
        if "tools" in request_lower or "help" in request_lower:
            tool_list = ", ".join(self.tools.keys())
            return f"I have the following tools available: {tool_list}. You can ask me to classify an image."

        return "I'm not sure how to handle that request. Try asking me to classify an image or type 'help' for available tools."

    def classify(self, image):
        """
        Classify an image and return a natural language response.

        Args:
            image: numpy array of shape (32, 32, 3).

        Returns:
            A formatted string with the classification results.
        """
        result = self.tools["classify_image"](image)

        # Format natural language response
        top = result["predictions"][0]
        response = f"I believe this is a **{top['class']}** (confidence: {top['confidence']:.1%}).\n"
        response += "\nTop-3 predictions:\n"
        for i, pred in enumerate(result["predictions"], 1):
            response += f"  {i}. {pred['class']}: {pred['confidence']:.1%}\n"

        return response


# Create the agent
agent = ImageClassifierAgent(model=model, class_names=CLASS_NAMES)
print("Agent created successfully.")
print(agent.process_request("help"))

## Step 5: Test the Agent

Let us test the agent on several CIFAR-10 test images and see how it responds.

In [ ]:
# Test the agent on multiple images
test_indices = [0, 1, 2, 3, 4, 100, 200, 500]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, idx in enumerate(test_indices):
    image = x_test[idx]
    true_label = CLASS_NAMES[int(y_test[idx])]

    # Get agent response
    response = agent.process_request("What is this image?", image=image)

    # Display image
    axes[i].imshow(image)
    axes[i].set_title(f"True: {true_label}", fontsize=10)
    axes[i].axis("off")

    # Print agent response
    print(f"--- Image {idx} (True: {true_label}) ---")
    print(response)

plt.suptitle("Agent Test Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Add Confidence Thresholds

A good agent should know when it is uncertain. We update the agent to:
- Say **"I'm not sure"** when the top confidence is below 0.5.
- Provide a `describe_prediction()` method that gives reasoning about the classification.

In [ ]:
class ImageClassifierAgent:
    """An AI agent that classifies images using a CNN model, with confidence thresholds."""

    CONFIDENCE_THRESHOLD = 0.5

    def __init__(self, model, class_names):
        self.model = model
        self.class_names = class_names
        self.tools = {
            "classify_image": self._classify_image_tool,
        }

    def _classify_image_tool(self, image):
        return classify_image(image, model=self.model, class_names=self.class_names)

    def process_request(self, request, image=None):
        request_lower = request.lower()

        if any(kw in request_lower for kw in ["classify", "identify", "what is", "recognise", "recognize"]):
            if image is None:
                return "I need an image to classify. Please provide one."
            return self.classify(image)

        if "tools" in request_lower or "help" in request_lower:
            tool_list = ", ".join(self.tools.keys())
            return f"Available tools: {tool_list}. Ask me to classify an image."

        return "I'm not sure how to handle that request. Try 'classify this image' or 'help'."

    def classify(self, image):
        result = self.tools["classify_image"](image)
        top = result["predictions"][0]

        if top["confidence"] < self.CONFIDENCE_THRESHOLD:
            response = f"I'm not sure about this one. My best guess is **{top['class']}**, "
            response += f"but my confidence is only {top['confidence']:.1%}.\n"
            response += "The image might be ambiguous or unlike anything I was trained on.\n"
        else:
            response = f"I believe this is a **{top['class']}** (confidence: {top['confidence']:.1%}).\n"

        response += "\nTop-3 predictions:\n"
        for i, pred in enumerate(result["predictions"], 1):
            response += f"  {i}. {pred['class']}: {pred['confidence']:.1%}\n"

        response += "\n" + self.describe_prediction(result)
        return response

    def describe_prediction(self, result):
        """
        Provide reasoning about the classification.

        Args:
            result: dict from classify_image.

        Returns:
            A string explaining the prediction.
        """
        preds = result["predictions"]
        top = preds[0]
        reasoning = "Reasoning: "

        if top["confidence"] > 0.9:
            reasoning += f"The model is very confident this is a {top['class']}. "
            reasoning += "The features strongly match this category."
        elif top["confidence"] > 0.7:
            reasoning += f"The model is fairly confident this is a {top['class']}. "
            if len(preds) > 1:
                reasoning += f"It could also be a {preds[1]['class']} ({preds[1]['confidence']:.1%})."
        elif top["confidence"] > 0.5:
            reasoning += f"The model leans towards {top['class']}, but there is some uncertainty. "
            if len(preds) > 1:
                gap = top["confidence"] - preds[1]["confidence"]
                reasoning += f"The gap to the next prediction ({preds[1]['class']}) is {gap:.1%}."
        else:
            reasoning += "The model is uncertain. Multiple classes have similar scores, "
            reasoning += "suggesting the image is ambiguous or does not clearly match any learned category."

        return reasoning


# Recreate the agent with the updated class
agent = ImageClassifierAgent(model=model, class_names=CLASS_NAMES)

# Test with several images
print("=== Testing updated agent with confidence thresholds ===\n")
for idx in [0, 10, 50, 150, 300]:
    image = x_test[idx]
    true_label = CLASS_NAMES[int(y_test[idx])]
    print(f"--- Image {idx} (True: {true_label}) ---")
    print(agent.process_request("Classify this image", image=image))
    print()

## Step 7: Vibe-Coding Challenge

Now it is your turn to extend the agent using the vibe-coding workflow.

**Challenge:** Add a second tool to the agent. Here are some ideas:

- **`find_similar`** -- Given an image, find the most similar images in the test set using cosine similarity on the CNN's feature embeddings.
- **`batch_classify`** -- Classify a batch of images at once and return a summary.
- **`explain_confidence`** -- Generate a more detailed explanation of why the model made a particular prediction.

**Vibe-coding workflow:**
1. **Prompt** an AI assistant: "Add a `find_similar` tool to the ImageClassifierAgent that finds the 5 most similar images from the test set."
2. **Generate** the code.
3. **Test** it by running the agent.
4. **Refine** until it works well.

Use the cell below to implement your solution.

In [ ]:
# Your vibe-coding challenge solution goes here.
# Use the vibe-coding workflow: Prompt -> Generate -> Test -> Refine

# Example prompt to give an AI assistant:
# "Add a find_similar tool to the ImageClassifierAgent that extracts features
#  from the CNN's second-to-last layer, computes cosine similarity against the
#  test set, and returns the 5 most similar images with their labels."

print("Implement your vibe-coding challenge here!")